# SIFLOW · Notebook 1 · MDLM full study (Table 2 main + Table 3)

Runs the entire primary study on the MDLM-170M teacher on a single A100-40GB: tokenize OpenWebText, train the velocity head, evaluate the k-sweep + teacher step-curve + AR GPT-2 (+ optional SDTT@8), build all figures, then the ablation suite — and auto-fills the LaTeX tables. Downloads `nb1_mdlm_outputs.zip` for Notebook 2.

**Runtime:** designed to finish in one Colab session on a single **A100-40GB**. Every stage is checkpointed and guarded by an existence check, and training has an 11h wall-clock guard — if a session ends early, just re-run this notebook (re-upload its own zip) and it skips finished stages and resumes training.

**How to use:** run every cell top-to-bottom. Cells 1–2 set up the environment and the artifact location; the last cell downloads this part's output zip.

In [ ]:
# === 1. Clone + install (run once per session, ~2 min) ===
REPO_URL = "https://github.com/kagtgi/siflow.git"
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# === 2. Where do artifacts live? ===
# Default: a local folder + zip download/upload between the 2 notebooks (no Drive).
# Set USE_DRIVE = True to persist on Google Drive instead (the import + download
# steps then become no-ops and everything survives a disconnect automatically).
USE_DRIVE = False

import os
if USE_DRIVE:
    from siflow.utils import drive
    drive.mount()
    os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
    BASE = drive.base_dir()
else:
    BASE = "/content/artifacts"
    os.makedirs(BASE, exist_ok=True)
print("artifacts ->", BASE)

In [ ]:
# === Quick smoke: unit tests + MDLM load + SUBS sanity ===
!python -m pytest tests/ -q

In [ ]:
import torch
from siflow.teacher import MDLMTeacher
teacher = MDLMTeacher(dtype=torch.bfloat16)
ids = torch.full((2, 32), teacher.mask_index, device=teacher.device)
z, _ = teacher.logits_and_hidden(ids)
print("P(mask) on mask token (should be ~0):",
      torch.softmax(z.float(), -1)[..., teacher.mask_index].max().item())
del teacher, z
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("smoke OK")

In [ ]:
# === Sizes (shrink any of these for a fast end-to-end smoke) ===
N_TRAIN   = 160_000   # training sequences (set 20_000 for a quick smoke)
N_VAL     = 5_000
STEPS     = 12000     # MDLM head training steps
ABL_STEPS = 3000      # steps per ablation variant
print("sizes set")

In [ ]:
# === Data: tokenize OpenWebText (train + disjoint val for MAUVE) ===
import os
from transformers import AutoTokenizer
from siflow.data import build_token_chunks
tok = AutoTokenizer.from_pretrained("gpt2")
if not os.path.exists(f"{BASE}/data/owt_gpt2_256.npy"):
    build_token_chunks(tok, 256, N_TRAIN, f"{BASE}/data/owt_gpt2_256.npy",
                       dataset="Skylion007/openwebtext", split="train")
if not os.path.exists(f"{BASE}/data/owt_gpt2_val.npy"):
    build_token_chunks(tok, 256, N_VAL, f"{BASE}/data/owt_gpt2_val.npy",
                       dataset="Skylion007/openwebtext", split="train", skip_seqs=N_TRAIN)
print("data ready")

In [ ]:
# === Train the MDLM velocity head (auto-resumes from checkpoint) ===
ip = get_ipython()
ip.system(f"python scripts/train.py --config siflow/config/mdlm.yaml --set "
          f"data.tokens_path={BASE}/data/owt_gpt2_256.npy "
          f"out_dir={BASE}/ckpt/mdlm run_id=siflow_mdlm train.total_steps={STEPS}")

In [ ]:
# === Training curves (skips cleanly if the log isn't there yet) ===
import os
_log = f"{BASE}/ckpt/mdlm/train_log.jsonl"
if os.path.exists(_log):
    from siflow.analysis.curves import load_jsonl, series
    import matplotlib.pyplot as plt
    rows = load_jsonl(_log)
    for k in ("satd", "vel", "mdm"):
        xs, ys = series(rows, k)
        if xs: plt.plot(xs, ys, label=k)
    plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.title("MDLM head training"); plt.show()
else:
    print("no train log at", _log)

In [ ]:
# === Evaluate SIFLOW (k-sweep), MDLM teacher step-curve, and AR GPT-2 ===
import os
ip = get_ipython()
if not os.path.exists(f"{BASE}/results/mdlm.json"):
    ip.system(f"python scripts/evaluate.py --config siflow/config/mdlm.yaml --system siflow "
              f"--ckpt-dir {BASE}/ckpt/mdlm --ref-tokens {BASE}/data/owt_gpt2_val.npy "
              f"--n-samples 1000 --k-list 1 2 4 8 --straightness-n 128 --out {BASE}/results/mdlm.json")
if not os.path.exists(f"{BASE}/results/mdlm_teacher.json"):
    ip.system(f"python scripts/evaluate.py --config siflow/config/mdlm.yaml --system teacher "
              f"--steps 8 32 64 1024 --ref-tokens {BASE}/data/owt_gpt2_val.npy "
              f"--n-samples 1000 --out {BASE}/results/mdlm_teacher.json")
if not os.path.exists(f"{BASE}/results/ar_gpt2.json"):
    ip.system(f"python scripts/evaluate.py --config siflow/config/mdlm.yaml --system ar --ar-model gpt2 "
              f"--ref-tokens {BASE}/data/owt_gpt2_val.npy --n-samples 1000 --out {BASE}/results/ar_gpt2.json")
print("eval done")

In [ ]:
# === (optional) SDTT@8 baseline -- skips gracefully if unavailable ===
import os
if not os.path.exists(f"{BASE}/results/sdtt.json"):
    try:
        !pip -q install git+https://github.com/jdeschena/sdtt.git
        import torch, json
        from sdtt import load_small_student
        from siflow.eval import decode_ids
        from siflow.eval.gen_ppl import GPT2Scorer
        from siflow.eval.diversity import diversity_metrics
        from transformers import AutoTokenizer
        m = load_small_student(loss="kld", round=7).cuda().eval()
        tok = AutoTokenizer.from_pretrained("gpt2"); texts = []
        while len(texts) < 1000:
            s = m.sample(n_samples=64, num_steps=8, seq_len=256)
            texts += decode_ids(s if torch.is_tensor(s) else torch.tensor(s), tok)
        sc = GPT2Scorer("gpt2-large")
        json.dump({"run_id": "sdtt", "method": "SDTT", "source": "reproduced",
                   "metrics": {"steps=8": {**sc.perplexity(texts), **diversity_metrics(texts), "nfe": 8}}},
                  open(f"{BASE}/results/sdtt.json", "w"), indent=2)
        print("SDTT done")
    except Exception as e:
        print("SDTT skipped:", e)

In [ ]:
# === Figures + tables (Table 2 main rows so far) ===
ip = get_ipython()
ip.system(f"python scripts/make_figures.py --results {BASE}/results "
          f"--train-log {BASE}/ckpt/mdlm/train_log.jsonl --out-dir {BASE}/figures")
ip.system(f"python scripts/make_tables.py --results {BASE}/results --out {BASE}/tables_auto.tex")
print(open(f"{BASE}/tables_auto.tex").read()[:1500])

In [ ]:
# === Save + auto-download this part's output ===
from siflow.utils.io import export_and_download
if not USE_DRIVE:
    export_and_download(BASE, 'nb1_mdlm_outputs.zip', include=['results', 'figures', 'tables_auto.tex', 'ckpt/mdlm'])
else:
    print('USE_DRIVE: outputs already persisted under', BASE)

In [ ]:
# === Ablations (Table 3) -- each variant is guarded + resumable ===
import os
ip = get_ipython()
ABLATIONS = {
  "abl_no_avg_velocity": "ablation.no_avg_velocity=true",
  "abl_hard_label":      "ablation.hard_label=true",
  "abl_no_entropy_prior":"train.lam_ent=0.0",
  "abl_identity_target": "train.w_id=0.5",
  "abl_prob_space":      "head.space=prob",
  "abl_head_depth1":     "head.mid_blocks=1",
}
for rid, override in ABLATIONS.items():
    if os.path.exists(f"{BASE}/results/{rid}.json"):
        print("skip (done):", rid); continue
    print("=== train", rid, "===")
    ip.system(f"python scripts/train.py --config siflow/config/mdlm.yaml --set "
              f"data.tokens_path={BASE}/data/owt_gpt2_256.npy out_dir={BASE}/ckpt/{rid} "
              f"run_id={rid} train.total_steps={ABL_STEPS} train.max_hours=1.0 {override}")
    ip.system(f"python scripts/evaluate.py --config siflow/config/mdlm.yaml --system siflow "
              f"--ckpt-dir {BASE}/ckpt/{rid} --ref-tokens {BASE}/data/owt_gpt2_val.npy "
              f"--n-samples 400 --k-list 1 8 --straightness-n 0 --set run_id={rid} "
              f"--out {BASE}/results/{rid}.json")
print("ablations done")

In [ ]:
# === Regenerate tables (now with the ablation rows) + re-export ===
ip = get_ipython()
ip.system(f"python scripts/make_tables.py --results {BASE}/results --out {BASE}/tables_auto.tex")
print(open(f"{BASE}/tables_auto.tex").read())

In [ ]:
# === Save + auto-download this part's output ===
from siflow.utils.io import export_and_download
if not USE_DRIVE:
    export_and_download(BASE, 'nb1_mdlm_outputs.zip', include=['results', 'figures', 'tables_auto.tex', 'ckpt/mdlm'])
else:
    print('USE_DRIVE: outputs already persisted under', BASE)

**Done with Notebook 1.** Keep `nb1_mdlm_outputs.zip` — Notebook 2 imports it so the final tables include these primary (MDLM) + ablation rows.